In [1]:
import re
import pandas as pd

# Define the file path
file_path = '20241023_PlasmaExtract1.txt'

# Initialize lists to store parsed data
filenames = []
q1_values = []
q3_values = []
summed_intensities = []
lipids = []

# Open and read the file
with open(file_path, 'r') as file:
    lines = file.readlines()

    current_filename = ""
    current_q1 = None
    current_q3 = None
    current_intensity_sum = 0
    current_lipid = ""
    parsing_intensity = False

    for line in lines:
        # Extract the filename (assumes filename appears earlier in the file)
        if 'sourceFile:' in line or 'name:' in line:
            match = re.search(r'name:\s+([\w.]+)', line)
            if match:
                current_filename = match.group(1)
        
        # Find Q1, Q3 values, and Lipid
        if 'id: SRM SIC Q1=' in line:
            match = re.search(r'Q1=(\d+\.\d+).*Q3=(\d+\.\d+).*name=([^\s]+)', line)
            if match:
                current_q1 = float(match.group(1))
                current_q3 = float(match.group(2))
                current_lipid = match.group(3)

        # Check if we are parsing intensity array data
        if 'cvParam: intensity array' in line:
            parsing_intensity = True
            current_intensity_sum = 0
        elif parsing_intensity and 'binary: [' in line:
            # Extract and sum intensity values
            match = re.search(r'binary:\s+\[\d+\]\s+([\d\s]+)', line)
            if match:
                intensities = map(int, match.group(1).split())
                current_intensity_sum = sum(intensities)
            parsing_intensity = False
            
            # Append the extracted data to the lists
            if current_filename and current_q1 is not None and current_q3 is not None:
                filenames.append(current_filename)
                q1_values.append(current_q1)
                q3_values.append(current_q3)
                summed_intensities.append(current_intensity_sum)
                lipids.append(current_lipid)

# Create a DataFrame
chromatogram_df = pd.DataFrame({
    'Lipid': lipids,
    'Q1': q1_values,
    'Q3': q3_values,
    'Intensity': summed_intensities,
    'Filename': filenames,
})

# Display the DataFrame
chromatogram_df


FileNotFoundError: [Errno 2] No such file or directory: '20241023_PlasmaExtract1.txt'

In [10]:
# sort by max Intensity
chromatogram_df.sort_values(by='Intensity', ascending=False, inplace=True)
chromatogram_df.reset_index(drop=True, inplace=True)
#print unique Lipid names
print(chromatogram_df['Lipid'].unique())
chromatogram_df

['CAR(10:2)_QUAL' 'CAR(2:0)' 'CAR_QUAL' 'CAR(20:4)' 'CAR(14:2)_QUAL'
 'CAR(8:1)' 'CAR(6:0)' 'CAR(10:1)' 'CAR(14:1)' 'CAR(10:0)' 'CAR(16:1)'
 'CAR(18:1)' 'CAR(8:0)' 'CAR' 'CAR(8:1)_QUAL' 'CAR(16:0)' 'CAR(18:3)'
 'CAR(18:2)' 'CAR(10:2)' 'CAR(26:0)' 'CAR(14:2)' 'CAR(9:0)' 'CAR(18:0)'
 'CAR(14:0)' 'CAR(11:0)' 'CAR(20:1)' 'CAR(5:0)' 'CAR(12:0)' 'CAR(20:2)'
 'CAR(3:0)' 'CAR(6:0)_QUAL' 'CAR(22:0)' 'CAR(10:3)' 'CAR(4:0)' 'CAR(17:0)'
 'CAR(20:0)' 'CAR(20:4)_QUAL' 'CAR(7:0)' 'CAR(6:1)_QUAL' 'CAR(22:2)'
 'CAR(16:2)' 'CAR(18:4)' 'CAR(6:1)' 'CAR(5:1)_QUAL' 'CAR(17:0)_QUAL'
 'CAR(2:0)_QUAL' 'CAR(22:6)' 'CAR(16:1)_QUAL' 'CAR(10:0)_QUAL' 'CAR(5:1)'
 'CAR(9:0)_QUAL' 'CAR(22:4)' 'CAR(4:1)' 'CAR(14:1)_QUAL' 'CAR(16:0)_QUAL'
 'CAR(4:1)_QUAL' 'CAR(10:1)_QUAL' 'CAR(8:0)_QUAL' 'CAR(3:1)'
 'CAR(18:1)_QUAL' 'CAR(11:0)_QUAL' 'CAR(22:5)' 'CAR(18:0)_QUAL'
 'CAR(18:3)_QUAL' 'CAR(10:3)_QUAL' 'CAR(14:0)_QUAL' 'CAR(26:0)_QUAL'
 'CAR(5:0)_QUAL' 'CAR(18:2)_QUAL' 'CAR(12:0)_QUAL' 'CAR(16:2)_QUAL'
 'CAR(22:2)_QUAL' 'CAR(

,Lipid,Q1,Q3,Intensity,Filename
0,CAR(10:2)_QUAL,312.22,60.1,34100,20241023_PlasmaExtract1.wiff.scan
1,CAR(2:0),204.12,85.1,29750,20241023_PlasmaExtract1.wiff.scan
2,CAR_QUAL,162.20,60.1,27825,20241023_PlasmaExtract1.wiff.scan
3,CAR(20:4),448.34,85.1,22975,20241023_PlasmaExtract1.wiff.scan
4,CAR(14:2)_QUAL,368.28,60.1,21950,20241023_PlasmaExtract1.wiff.scan
...,...,...,...,...,...
79,CAR(7:0)_QUAL,274.20,60.1,150,20241023_PlasmaExtract1.wiff.scan
80,CAR(20:0)_QUAL,456.41,60.1,125,20241023_PlasmaExtract1.wiff.scan
81,CAR(22:6)_QUAL,472.40,60.1,125,20241023_PlasmaExtract1.wiff.scan
82,CAR(20:2)_QUAL,452.37,60.1,125,20241023_PlasmaExtract1.wiff.scan


In [2]:
import re
import os
import pandas as pd

# Set the directory paths for input and output
input_dir = 'text/'
output_dir = 'result'
output_file = os.path.join(output_dir, 'parsed_chromatogram_data.csv')

# Initialize lists to store parsed data
filenames = []
q1_values = []
q3_values = []
summed_intensities = []
lipids = []

# Iterate through all .txt files in the specified directory
for file_name in os.listdir(input_dir):
    if file_name.endswith('.txt'):
        file_path = os.path.join(input_dir, file_name)
        
        # Open and read the file
        with open(file_path, 'r') as file:
            lines = file.readlines()

            current_filename = ""
            current_q1 = None
            current_q3 = None
            current_intensity_sum = 0
            current_lipid = ""
            parsing_intensity = False

            for line in lines:
                # Extract the filename (assumes filename appears earlier in the file)
                if 'sourceFile:' in line or 'name:' in line:
                    match = re.search(r'name:\s+([\w.]+)', line)
                    if match:
                        current_filename = match.group(1)
                
                # Find Q1, Q3 values, and Lipid
                if 'id: SRM SIC Q1=' in line:
                    match = re.search(r'Q1=(\d+\.\d+).*Q3=(\d+\.\d+).*name=([^\s]+)', line)
                    if match:
                        current_q1 = float(match.group(1))
                        current_q3 = float(match.group(2))
                        current_lipid = match.group(3)

                # Check if we are parsing intensity array data
                if 'cvParam: intensity array' in line:
                    parsing_intensity = True
                    current_intensity_sum = 0
                elif parsing_intensity and 'binary: [' in line:
                    # Extract and sum intensity values
                    match = re.search(r'binary:\s+\[\d+\]\s+([\d\s]+)', line)
                    if match:
                        intensities = map(int, match.group(1).split())
                        current_intensity_sum = sum(intensities)
                    parsing_intensity = False
                    
                    # Append the extracted data to the lists
                    if current_filename and current_q1 is not None and current_q3 is not None:
                        filenames.append(current_filename)
                        q1_values.append(current_q1)
                        q3_values.append(current_q3)
                        summed_intensities.append(current_intensity_sum)
                        lipids.append(current_lipid)

# Create a DataFrame
chromatogram_df = pd.DataFrame({
    'Lipid': lipids,
    'Q1': q1_values,
    'Q3': q3_values,
    'Intensity': summed_intensities,
    'Filename': filenames,
})

# Save the DataFrame to a CSV file
chromatogram_df.to_csv(output_file, index=False)

# Display the DataFrame
print(chromatogram_df)

                        Lipid      Q1      Q3  Intensity  \
0                   LPE(12:0)  398.23  257.23          0   
1                   LPE(14:0)  426.26  285.26         40   
2                   LPE(14:1)  424.25  283.25         40   
3     "LPE(15:1),LPE(P-16:0)"  438.30  297.30          0   
4                   LPE(16:1)  452.28  311.28          0   
...                       ...     ...     ...        ...   
3888           DG(44:4)_C18:0  746.67  445.37        800   
3889           DG(44:3)_C18:0  748.68  447.38        520   
3890           DG(44:2)_C18:0  750.70  449.40        160   
3891           DG(44:1)_C18:0  752.71  451.41        720   
3892           DG(44:0)_C18:0  754.73  453.43        320   

                                Filename  
0     20241023_PlasmaExtract11.wiff.scan  
1     20241023_PlasmaExtract11.wiff.scan  
2     20241023_PlasmaExtract11.wiff.scan  
3     20241023_PlasmaExtract11.wiff.scan  
4     20241023_PlasmaExtract11.wiff.scan  
...                  

# call python file

In [3]:
# Import the function from QTRAP.py
from QTRAP import parse_chromatogram_data

# Define input and output directories
input_dir = 'text/'
output_dir = 'result/'

# Parse the chromatogram data and save to a CSV
chromatogram_df = parse_chromatogram_data(input_dir, output_dir)

# Display the resulting DataFrame
chromatogram_df


,Lipid,Q1,Q3,Intensity,Filename
0,LPE(12:0),398.23,257.23,0,20241023_PlasmaExtract11.wiff.scan
1,LPE(14:0),426.26,285.26,40,20241023_PlasmaExtract11.wiff.scan
2,LPE(14:1),424.25,283.25,40,20241023_PlasmaExtract11.wiff.scan
3,"""LPE(15:1),LPE(P-16:0)""",438.30,297.30,0,20241023_PlasmaExtract11.wiff.scan
4,LPE(16:1),452.28,311.28,0,20241023_PlasmaExtract11.wiff.scan
...,...,...,...,...,...
3888,DG(44:4)_C18:0,746.67,445.37,800,20241023_PlasmaExtract6.wiff.scan
3889,DG(44:3)_C18:0,748.68,447.38,520,20241023_PlasmaExtract6.wiff.scan
3890,DG(44:2)_C18:0,750.70,449.40,160,20241023_PlasmaExtract6.wiff.scan
3891,DG(44:1)_C18:0,752.71,451.41,720,20241023_PlasmaExtract6.wiff.scan


In [2]:
from QTRAP_plot import plot_lipid_intensities

# Define input CSV and output directory for plots
input_csv = 'result/parsed_chromatogram_data.csv'
output_dir = 'result/plots/'

# # Generate bar plots sorted by Lipid
# plot_lipid_intensities(input_csv, output_dir, sorting='Lipid')

# Generate bar plots sorted by Intensity
plot_lipid_intensities(input_csv, output_dir, sorting='Intensity')


Processing files:   0%|          | 0/30 [00:00<?, ?it/s]

Plot saved: result/plots/20241023_Blank31.wiff.scan/20241023_Blank31.wiff.scan_chunk_1_lipid_intensities.png
Plot saved: result/plots/20241023_Blank31.wiff.scan/20241023_Blank31.wiff.scan_chunk_2_lipid_intensities.png
Plot saved: result/plots/20241023_Blank31.wiff.scan/20241023_Blank31.wiff.scan_chunk_3_lipid_intensities.png
Plot saved: result/plots/20241023_Blank31.wiff.scan/20241023_Blank31.wiff.scan_chunk_4_lipid_intensities.png
Plot saved: result/plots/20241023_Blank31.wiff.scan/20241023_Blank31.wiff.scan_chunk_5_lipid_intensities.png
Plot saved: result/plots/20241023_Blank31.wiff.scan/20241023_Blank31.wiff.scan_chunk_6_lipid_intensities.png
Plot saved: result/plots/20241023_Blank31.wiff.scan/20241023_Blank31.wiff.scan_chunk_7_lipid_intensities.png
Plot saved: result/plots/20241023_Blank31.wiff.scan/20241023_Blank31.wiff.scan_chunk_8_lipid_intensities.png


Processing files:   3%|▎         | 1/30 [00:01<00:37,  1.31s/it]

Plot saved: result/plots/20241023_Blank31.wiff.scan/20241023_Blank31.wiff.scan_chunk_9_lipid_intensities.png
Plot saved: result/plots/20241023_Blank31.wiff.scan/20241023_Blank31.wiff.scan_chunk_10_lipid_intensities.png
Plot saved: result/plots/20241023_Blank31.wiff.scan/20241023_Blank31.wiff.scan_chunk_11_lipid_intensities.png
Plot saved: result/plots/20241023_Blank32.wiff.scan/20241023_Blank32.wiff.scan_chunk_1_lipid_intensities.png
Plot saved: result/plots/20241023_Blank32.wiff.scan/20241023_Blank32.wiff.scan_chunk_2_lipid_intensities.png
Plot saved: result/plots/20241023_Blank32.wiff.scan/20241023_Blank32.wiff.scan_chunk_3_lipid_intensities.png
Plot saved: result/plots/20241023_Blank32.wiff.scan/20241023_Blank32.wiff.scan_chunk_4_lipid_intensities.png
Plot saved: result/plots/20241023_Blank32.wiff.scan/20241023_Blank32.wiff.scan_chunk_5_lipid_intensities.png
Plot saved: result/plots/20241023_Blank32.wiff.scan/20241023_Blank32.wiff.scan_chunk_6_lipid_intensities.png
Plot saved: resul

Processing files:   7%|▋         | 2/30 [00:02<00:32,  1.15s/it]

...

Plot saved: result/plots/20241023_PlasmaExtract7.wiff.scan/20241023_PlasmaExtract7.wiff.scan_chunk_9_lipid_intensities.png
Plot saved: result/plots/20241023_PlasmaExtract7.wiff.scan/20241023_PlasmaExtract7.wiff.scan_chunk_10_lipid_intensities.png
Plot saved: result/plots/20241023_PlasmaExtract8.wiff.scan/20241023_PlasmaExtract8.wiff.scan_chunk_1_lipid_intensities.png
Plot saved: result/plots/20241023_PlasmaExtract8.wiff.scan/20241023_PlasmaExtract8.wiff.scan_chunk_2_lipid_intensities.png
Plot saved: result/plots/20241023_PlasmaExtract8.wiff.scan/20241023_PlasmaExtract8.wiff.scan_chunk_3_lipid_intensities.png
Plot saved: result/plots/20241023_PlasmaExtract8.wiff.scan/20241023_PlasmaExtract8.wiff.scan_chunk_4_lipid_intensities.png
Plot saved: result/plots/20241023_PlasmaExtract8.wiff.scan/20241023_PlasmaExtract8.wiff.scan_chunk_5_lipid_intensities.png
Plot saved: result/plots/20241023_PlasmaExtract8.wiff.scan/20241023_PlasmaExtract8.wiff.scan_chunk_6_lipid_intensities.png
Plot saved: res

Processing files:  97%|█████████▋| 29/30 [00:42<00:01,  1.05s/it]

Plot saved: result/plots/20241023_PlasmaExtract8.wiff.scan/20241023_PlasmaExtract8.wiff.scan_chunk_8_lipid_intensities.png
Plot saved: result/plots/20241023_PlasmaExtract8.wiff.scan/20241023_PlasmaExtract8.wiff.scan_chunk_9_lipid_intensities.png
Plot saved: result/plots/20241023_PlasmaExtract9.wiff.scan/20241023_PlasmaExtract9.wiff.scan_chunk_1_lipid_intensities.png
Plot saved: result/plots/20241023_PlasmaExtract9.wiff.scan/20241023_PlasmaExtract9.wiff.scan_chunk_2_lipid_intensities.png
Plot saved: result/plots/20241023_PlasmaExtract9.wiff.scan/20241023_PlasmaExtract9.wiff.scan_chunk_3_lipid_intensities.png
Plot saved: result/plots/20241023_PlasmaExtract9.wiff.scan/20241023_PlasmaExtract9.wiff.scan_chunk_4_lipid_intensities.png
Plot saved: result/plots/20241023_PlasmaExtract9.wiff.scan/20241023_PlasmaExtract9.wiff.scan_chunk_5_lipid_intensities.png
Plot saved: result/plots/20241023_PlasmaExtract9.wiff.scan/20241023_PlasmaExtract9.wiff.scan_chunk_6_lipid_intensities.png
Plot saved: resu

Processing files: 100%|██████████| 30/30 [00:44<00:00,  1.50s/it]

Plot saved: result/plots/20241023_PlasmaExtract9.wiff.scan/20241023_PlasmaExtract9.wiff.scan_chunk_18_lipid_intensities.png


In [4]:
# print uniqu filename
print(chromatogram_df['Filename'].unique())

['20241023_PlasmaExtract11.wiff.scan' '20241023_Blank31.wiff.scan'
 '20241023_PlasmaExtract13.wiff.scan' '20241023_PlasmaExtract8.wiff.scan'
 '20241023_PlasmaExtract14.wiff.scan' '20241023_PlasmaExtract4.wiff.scan'
 '20241023_PlasmaExtract3.wiff.scan' '20241023_PlasmaExtract12.wiff.scan'
 '20241023_PlasmaExtract21.wiff.scan' '20241023_PlasmaExtract24.wiff.scan'
 '20241023_PlasmaExtract16.wiff.scan' '20241023_PlasmaExtract20.wiff.scan'
 '20241023_PlasmaExtract17.wiff.scan' '20241023_PlasmaExtract23.wiff.scan'
 '20241023_PlasmaExtract15.wiff.scan' '20241023_PlasmaExtract10.wiff.scan'
 '20241023_PlasmaExtract25.wiff.scan' '20241023_PlasmaExtract19.wiff.scan'
 '20241023_PlasmaExtract5.wiff.scan' '20241023_PlasmaExtract18.wiff.scan'
 '20241023_Blank34.wiff.scan' '20241023_PlasmaExtract2.wiff.scan'
 '20241023_Blank33.wiff.scan' '20241023_PlasmaExtract9.wiff.scan'
 '20241023_Blank32.wiff.scan' '20241023_PlasmaExtract1.wiff.scan'
 '20241023_Blank35.wiff.scan' '20241023_PlasmaExtract7.wiff.scan